# Inference Pipeline
---
### GEE (S1 - S2 - DEM)  ·  GCS Tiles   ·  Local Inference  ·  COG   ·  Geojson  ·  CSV

## Configuration

### Imports

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import glob
import io
import json
import os
from pathlib import Path
import re
import time
import warnings

import ee
import geopandas as gpd
from google.cloud import storage as gcs
import joblib
import numpy as np
from osgeo import gdal
import pandas as pd
import rasterio
from rasterio.merge import merge
from rasterstats import zonal_stats
from rio_cogeo.cogeo import cog_translate
from rio_cogeo.profiles import cog_profiles
from tqdm.auto import tqdm

from indices import add_s2_indices, add_s1_indices

warnings.filterwarnings('ignore')

print("Imports loaded.")


### Constants

In [ ]:
# GCS
PERIOD = "2023-01"
BUCKET_NAME = "carbonlens-bucket"
GCS_TILES_PREFIX = "v1/senegal/feature_tiles"
GCS_COG_PREFIX = "v1/senegal/maps"
GCS_REGIONS_PREFIX = "v1/senegal/regions"
GCS_DEPTS_PREFIX = "v1/senegal/departments"
GCS_COMS_PREFIX = "v1/senegal/communes"
GCS_PA_PREFIX = "v1/senegal/protected_areas"
GCS_COG_FILENAME = f"carbon_pred_{PERIOD}_10m_COG.tif"
GCS_COG_PATH = f"{GCS_COG_PREFIX}/{GCS_COG_FILENAME}"
COG_PATTERN = re.compile(r"carbon_pred_(\d{4}-\d{2}(?:-\d{2})?)_10m_COG\.tif$")

# Local paths
FOLDER = "data/sen"
CSV_DIR = f"{FOLDER}/csvs"
GEOJSON_DIR = f"{FOLDER}/geojsons"
LOCAL_WORK_DIR = "inference"
LOCAL_RAW_DIR = f"{LOCAL_WORK_DIR}/raw_tiles/"
LOCAL_PRED_DIR = f"{LOCAL_WORK_DIR}/pred_tiles/"
COG_OUTPUT_PATH = f"{LOCAL_WORK_DIR}/{GCS_COG_FILENAME}"
GCP_KEY_PATH = "secrets/gcp-sa-key.json"
SENEGAL_GEOJSON = f"{FOLDER}/gadm41_SEN_0.json"
BASE_REGIONS_PATH = "data/sen/senegal_regions_preds_2024-05.geojson"
BASE_DEPTS_PATH = "data/sen/senegal_departments_preds_2024-05.geojson"
BASE_COMS_PATH = "data/sen/senegal_communes_preds_2024-05.geojson"
BASE_PA_PATH = "data/sen/senegal_protected_areas_preds_2024-05.geojson"
OUT_REGIONS_PATH = f"{GEOJSON_DIR}/senegal_regions_preds_{PERIOD}.geojson"
OUT_DEPTS_PATH = f"{GEOJSON_DIR}/senegal_departments_preds_{PERIOD}.geojson"
OUT_COMS_PATH = f"{GEOJSON_DIR}/senegal_communes_preds_{PERIOD}.geojson"
OUT_PA_PATH = f"{GEOJSON_DIR}/senegal_protected_areas_preds_{PERIOD}.geojson"
LAND_AREA_PATH = f"{FOLDER}/land_area.txt"
MOSAIC_TMP_PATH = f"{LOCAL_WORK_DIR}/mosaic_tmp.tif"

# Model and features
FEATURES_FILE = f"{FOLDER}/selected_features.txt"
MODEL_DIR = "models"
BEST_MODEL_GLOB = "BEST_*.pkl"

# Inference parameters
INFERENCE_START = "2023-01-01"
INFERENCE_END = "2023-02-01"
EPS = 1e-9
S2_CLOUD_MAX = 15
GEE_SCALE = 10
CRSG = "EPSG:4326"
CRSP= "EPSG:3857"
NODATA_VAL= -9999
MAX_AGB   = 500
N_WORKERS = 4
TASK_POLL_SEC = 1
BINS = [0, 5, 10, 15, 20, 25, float("inf")]
REG_POS = 11
DEPT_POS = 13
COM_POS = 14
PA_POS = 33
BUCKET_PREFIX = "v1/senegal"
CSV_BLOB_NAME = f"{BUCKET_PREFIX}/data_index.csv"
BASE_URL = f"https://storage.googleapis.com/{BUCKET_NAME}/{GCS_COG_PREFIX}"
RES = "10m"
COUNTRY = "senegal"

# Bands
S1_BANDS  = ['VV', 'VH']
S2_BANDS  = ['B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12', 'B1']
DEM_BANDS = ['elevation']
ALL_BANDS = S1_BANDS + S2_BANDS + DEM_BANDS

# Working directories
for d in [CSV_DIR, GEOJSON_DIR , LOCAL_WORK_DIR, LOCAL_RAW_DIR, LOCAL_PRED_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")


### Authentification & Setup

In [ ]:
# Google Earth Engine
with open(GCP_KEY_PATH, "r", encoding="utf-8") as f:
    json_key = json.load(f)

credentials = ee.ServiceAccountCredentials(json_key['client_email'], GCP_KEY_PATH)
ee.Initialize(credentials)
print("Earth Engine initialized.")

# Google Cloud Storage
gcs_client = gcs.Client.from_service_account_json(GCP_KEY_PATH)
bucket = gcs_client.bucket(BUCKET_NAME)
print(f"GCS connected — bucket : gs://{BUCKET_NAME}")


### Model & Features Loading

In [ ]:
# Features
with open(FEATURES_FILE, "r", encoding="utf-8") as f:
    FEATURES = [line.strip() for line in f if line.strip()]

print(f"{len(FEATURES)} features loaded :")
for i, feat in enumerate(FEATURES, 1):
    print(f"   {i:>2}. {feat}")

# Model
model_files = sorted(glob.glob(os.path.join(MODEL_DIR, BEST_MODEL_GLOB)))
if not model_files:
    raise FileNotFoundError(f"No model found in {MODEL_DIR}")
best_model_path = model_files[-1]

print(f"\nModel path : {best_model_path}")


## Multi-band GEE image construction

### Senegal AOI

In [ ]:
# Senegal Bounds
gdf_senegal = gpd.read_file(SENEGAL_GEOJSON)
senegal_geom_wgs84 = gdf_senegal.union_all()
senegal_bounds = gdf_senegal.total_bounds

# Conversion in ee.Geometry for GEE
senegal_ee = ee.FeatureCollection(
    json.loads(gdf_senegal.to_json())
).geometry()

print("Senegal AOI (WGS84) :")
print(f"   lon : [{senegal_bounds[0]:.4f}°, {senegal_bounds[2]:.4f}°]")
print(f"   lat : [{senegal_bounds[1]:.4f}°, {senegal_bounds[3]:.4f}°]")
print(f"   Approximative area : {senegal_geom_wgs84.area * 111**2:.0f} km²")


### GEE preprocessing functions

In [ ]:
def mask_edge(image):
    """Mask S1 image edges using a -30 dB threshold."""
    edge = image.lt(-30.0)
    masked_image = image.mask().And(edge.Not())
    return image.updateMask(masked_image)

def mask_s2_clouds(image):
    """Mask clouds and cirrus from S2 QA60 band, and scale reflectances."""
    qa = image.select('QA60')
    cloud_bit_mask  = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )
    return image.updateMask(mask).divide(10000)

print("GEE preprocessing functions defined.")


### Composite S1 + S2 + DEM

In [ ]:
# Sentinel-1
s1_composite = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(senegal_ee)
    .filterDate(INFERENCE_START, INFERENCE_END)
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .map(mask_edge)
    .select(S1_BANDS)
    .mean()
    .clip(senegal_ee)
)

# Sentinel-2
s2_composite = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(senegal_ee)
    .filterDate(INFERENCE_START, INFERENCE_END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', S2_CLOUD_MAX))
    .map(mask_s2_clouds)
    .select(S2_BANDS)
    .median()
    .clip(senegal_ee)
)

# DEM SRTM reprojected to 10 m
dem = (
    ee.Image('USGS/SRTMGL1_003')
    .select('elevation')
    .reproject(crs=s2_composite.projection(), scale=GEE_SCALE)
    .clip(senegal_ee)
)

# Final Stack
feature_image = (
    ee.Image.cat([s1_composite, s2_composite, dem])
    .unmask(NODATA_VAL)
    .toFloat()
)

print(f"Image GEE built — {len(ALL_BANDS)} bands : {ALL_BANDS}")
print(f"Period : From {INFERENCE_START} to {INFERENCE_END}")


## GCS Export

### Export Task Submission to GCS

In [ ]:
def submit_export_task(image, bucket_name, prefix, aoi, scale, crs, nodata):
    """Submits a GEE export task."""

    task_in = ee.batch.Export.image.toCloudStorage(
        image = image,
        bucket = bucket_name,
        fileNamePrefix = f"{prefix}/",
        region = aoi,
        scale = scale,
        crs = crs,
        fileFormat = 'GeoTIFF',
        maxPixels = int(1e13),
        formatOptions  = {
            'cloudOptimized' : False,
            'noData'         : nodata,
        }
    )
    task_in.start()
    return task_in


task = submit_export_task(
    image = feature_image,
    bucket_name = BUCKET_NAME,
    prefix = GCS_TILES_PREFIX,
    aoi = senegal_ee,
    scale = GEE_SCALE,
    crs = CRSP,
    nodata = NODATA_VAL,
)
print(f"Destination : gs://{BUCKET_NAME}/{GCS_TILES_PREFIX}/")


## Export Task Monitoring

In [ ]:
def monitor_export_task(task_arg, check_interval=30, verbose=True):
    """Monitor an Earth Engine export task until completion or failure."""
    previous_state = None
    start_time = time.time()
    spin_chars = ['|', '/', '-', '\\']
    spin_idx = 0

    while True:
        status = task_arg.status()
        state = status['state']

        if verbose:
            # Display current state
            if state != previous_state:
                print()
                previous_state = state

            if state == 'READY':
                print("Task ready, waiting to start...", end='\r')
            elif state == 'RUNNING':
                # Duration elapsed
                elapsed = int(time.time() - start_time)
                minutes, seconds = divmod(elapsed, 60)
                # Display spinner
                spin = spin_chars[spin_idx % len(spin_chars)]
                print(f"Running... {spin} ({minutes:02d}:{seconds:02d})", end='\r')
                spin_idx += 1
            elif state == 'COMPLETED':
                print("\nTask completed successfully.")
                return True
            elif state == 'FAILED':
                error = status.get('error_message', 'Unknown error')
                raise Exception(f"Task failed: {error}")
            elif state == 'CANCELLED':
                raise Exception("Task was cancelled.")
            elif state == 'CANCEL_REQUESTED':
                print("Cancel requested, waiting...", end='\r')
            else:
                print(f"Unknown state: {state}", end='\r')

        # Exit loop
        if state in ('COMPLETED', 'FAILED', 'CANCELLED'):
            break

        # Wait before next check
        time.sleep(check_interval)

    return False


In [ ]:
def list_tiles_in_bucket(bucket_arg, bucket_name, prefix=None, extension='.tif'):
    """Lists all tile files (GeoTIFFs) in a GCS bucket, optionally filtered by a prefix."""
    blobs = bucket_arg.list_blobs(prefix=prefix)

    tiles = []
    for blob_in in blobs:
        # Filter by extension (case‑insensitive)
        blob_name = blob_in.name.split('/')[-1]
        if blob_name.lower().endswith(extension.lower()):
            tiles.append(blob_in.name)
    # Print summary
    if tiles:
        print(f"Found {len(tiles)} tile(s) in gs://{bucket_name}/{prefix or ''}")
    else:
        print(f"No tiles with extension '{extension}' found in gs://{bucket_name}/{prefix or ''}")

    return tiles


In [ ]:
# Monitor the task
try:
    monitor_export_task(task, check_interval=15)
    print("Export finished. Proceed to next action.")
    available_tiles = list_tiles_in_bucket(
        bucket_arg=bucket,
        bucket_name=BUCKET_NAME,
        prefix=GCS_TILES_PREFIX,
        extension='.tif'
    )
except Exception as e:
    print(f"Export problem: {e}")

## Tiles & Inference

### Inference Function on Tile

In [ ]:
def infer_tile(raw_path, pred_path, model_path, features, all_bands, nodata, max_agb):
    """
    Apply the AGB model to a raw (multi-band) tile and save
    the prediction tile (single-band, Float32).
    """
    with rasterio.open(raw_path) as src:
        height, width = src.height, src.width
        profile = src.profile.copy()
        data = src.read().astype(np.float32)

    nodata_mask = np.all(data == nodata, axis=0)

    n_pixels = height * width
    df_tile = pd.DataFrame(
        data.reshape(len(all_bands), n_pixels).T,
        columns=all_bands
    )

    df_tile = add_s2_indices(df_tile)
    df_tile = add_s1_indices(df_tile)

    missing = [f for f in features if f not in df_tile.columns]
    if missing:
        raise ValueError(f"Missing features in {raw_path} : {missing}")

    X_tile = df_tile[features].values

    has_invalid_features = ~np.isfinite(X_tile).all(axis=1)

    valid_flat = ~nodata_mask.ravel() & ~has_invalid_features

    preds_flat = np.full(n_pixels, nodata, dtype=np.float32)
    if valid_flat.sum() > 0:
        model = joblib.load(model_path)
        raw_preds = model.predict(X_tile[valid_flat]).astype(np.float32)
        raw_preds = np.clip(raw_preds, 0.0, max_agb)
        preds_flat[valid_flat] = 0.47 * raw_preds

    pred_grid = preds_flat.reshape(1, height, width)
    profile.update(
        count=1, dtype='float32', nodata=nodata,
        compress='deflate', tiled=True,
        blockxsize=256, blockysize=256,
    )
    with rasterio.open(pred_path, 'w', **profile) as dst:
        dst.write(pred_grid)
        dst.update_tags(unit='tC/ha', source='AGB_inference')
    return pred_path

print("infer_tile defined.")


### Prediction on Tiles in Parallel

In [ ]:
def download_blob(bucket_arg, gcs_path, local_path):
    """Download a blob from the bucket."""
    blob_in = bucket_arg.blob(gcs_path)
    blob_in.download_to_filename(local_path)
    return local_path


def process_tile(tile_name):
    """Download and process a tile."""
    modified_tile_name = tile_name.split('/')[-1]
    raw_local = os.path.join(LOCAL_RAW_DIR,  f"{modified_tile_name}")
    pred_local = os.path.join(LOCAL_PRED_DIR, f"pred_{modified_tile_name}")

    try:
        download_blob(bucket, tile_name, raw_local)

        infer_tile(
            raw_path = raw_local,
            pred_path = pred_local,
            model_path = best_model_path,
            features = FEATURES,
            all_bands = ALL_BANDS,
            nodata = float(NODATA_VAL),
            max_agb = MAX_AGB,
        )

        os.remove(raw_local)

        return {'name': tile_name, 'status': 'OK', 'pred': pred_local}

    except Exception as e:
        return {'name': tile_name, 'status': 'ERROR', 'error': str(e)}


tile_names = sorted(available_tiles)

print(f"Inference on {len(tile_names)} tiles ({N_WORKERS} workers) ...\n")

inference_results = []

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(process_tile, name): name for name in tile_names}

    with tqdm(total=len(futures), desc="Inference", unit="tile") as pbar:
        for future in as_completed(futures):
            result = future.result()
            inference_results.append(result)
            if result['status'] == 'ERROR':
                tqdm.write(f"  {result['name']} → {result['error']}")
            pbar.update(1)

ok_tiles    = [r for r in inference_results if r['status'] == 'OK']
error_tiles = [r for r in inference_results if r['status'] == 'ERROR']
pred_paths  = [r['pred'] for r in ok_tiles]

print(f"\n{'─'*50}")
print(f"Inference with success : {len(ok_tiles)} / {len(tile_names)} tiles")
if error_tiles:
    print(f"Errors : {[r['name'] for r in error_tiles]}")
print(f"{'─'*50}")


## Final COG

### Tiles prediction Mosaicking

In [ ]:
print(f"Mosaic of {len(pred_paths)} tuiles ...")

# Prediction Tiles Opening
src_files = [rasterio.open(p) for p in pred_paths]

try:
    mosaic, mosaic_transform = merge(
        src_files,
        nodata = float(NODATA_VAL),
        method = 'first',
        precision = 6,
    )

    mosaic_profile = src_files[0].profile.copy()
    mosaic_profile.update(
        height = mosaic.shape[1],
        width = mosaic.shape[2],
        transform = mosaic_transform,
        count = 1,
        dtype = 'float32',
        nodata = float(NODATA_VAL),
        compress = 'deflate',
        tiled = True,
        blockxsize = 512,
        blockysize = 512,
        bigtiff = 'YES',
    )

    with rasterio.open(MOSAIC_TMP_PATH, 'w', **mosaic_profile) as dst:
        dst.write(mosaic)
        dst.update_tags(
            title = 'Senegal CAGB 10m',
            unit = 'tC/ha',
            crs = CRSP,
            period = f"{INFERENCE_START} to {INFERENCE_END}",
            model = Path(best_model_path).name,
            features = ', '.join(FEATURES),
            resolution = f"{GEE_SCALE}m",
        )

    print(f"Temporary mosaic saved : {MOSAIC_TMP_PATH}")
    print(f"   Dimensions : {mosaic.shape[2]} × {mosaic.shape[1]} pixels")
    print(f"   Footprint    : {mosaic_transform}")

finally:
    for s in src_files:
        s.close()


### Conversion in Cloud Optimized GeoTIFF (COG)

In [ ]:
print(f"Conversion in COG : {COG_OUTPUT_PATH} ...")

# Profil COG
cog_profile = cog_profiles.get("deflate")
cog_profile.update(
    blockxsize = 512,
    blockysize = 512,
    dtype = 'float32',
    nodata = float(NODATA_VAL),
    BIGTIFF = "IF_SAFER"
)

cog_translate(
    MOSAIC_TMP_PATH, COG_OUTPUT_PATH,
    cog_profile,
    overview_level=5, overview_resampling="nearest",
    nodata=NODATA_VAL, add_mask=False, web_optimized=True # add_mask=False to check
)

# Remove temporary file
os.remove(MOSAIC_TMP_PATH)

# COG validation
cog_size_mb = os.path.getsize(COG_OUTPUT_PATH) / (1024 ** 2)
print("\nCOG generated successfully !")
print(f"   Path  : {COG_OUTPUT_PATH}")
print(f"   Size  : {cog_size_mb:.1f} Mo")

# COG Structure validation
ds = gdal.Open(COG_OUTPUT_PATH)
print(f"   Bands  : {ds.RasterCount}")
print(f"   Pixels  : {ds.RasterXSize} × {ds.RasterYSize}")
# print(f"   CRS     : {ds.GetProjection()[:60]}...")
ovr_count = ds.GetRasterBand(1).GetOverviewCount()
print(f"   Overviews : {ovr_count} levels")
ds = None


### COG Upload to GCS

In [ ]:
print("COG Upload to GCS ...")
print(f"Source      : {COG_OUTPUT_PATH}")
print(f"Destination : gs://{BUCKET_NAME}/{GCS_COG_PATH}")

blob_cog = bucket.blob(GCS_COG_PATH)
blob_cog.chunk_size = 8 * 1024 * 1024

with tqdm(total=cog_size_mb, unit='Mo', desc='Upload COG', unit_scale=True) as pbar:
    blob_cog.upload_from_filename(
        COG_OUTPUT_PATH,
        content_type = 'image/tiff',
    )
    pbar.update(cog_size_mb)


print("\nCOG uploaded successfully !")
print(f"   URI GCS : gs://{BUCKET_NAME}/{GCS_COG_PATH}")


## Update Data Index CSV File

In [ ]:
# COG mean & landscape area
with rasterio.open(COG_OUTPUT_PATH) as dataset:
    data = dataset.read(1)
data = data[data != -9999]
CARBON_MEAN = np.round(np.mean(data), 2)
counts, _ = np.histogram(data, bins=BINS)
OL, S, YRF, LDF, MDF, HDF = counts[0], counts[1], counts[2], counts[3], counts[4], counts[5]
TOTAL = counts.sum()

with open(LAND_AREA_PATH, 'r', encoding="utf-8") as f:
    LAND_AREA = float([line.strip() for line in f if line.strip()][0])

print(f"Carbon mean : {CARBON_MEAN:.2f} t/ha")
print(f"Landscape area : {LAND_AREA} ha")
print(f"Pixel counts : {counts}")


In [ ]:
# Update Data Index CSV File
blob_csv = bucket.blob(CSV_BLOB_NAME)

if blob_csv.exists():
    csv_data = blob_csv.download_as_text()
    df = pd.read_csv(io.StringIO(csv_data))
else:
    df = pd.DataFrame(columns=[
        "date", "res", "country", "url", "carbon_mean", "land_area",
        "ol", "s", "yrf", "ldf", "mdf", "hdf", "total"
    ])

existing_dates = set(df["date"].astype(str))
new_rows = []

for blob in bucket.list_blobs(prefix=GCS_COG_PREFIX):
    fname = Path(blob.name).name
    match = COG_PATTERN.match(fname)
    if match:
        date = match.group(1)
        if date not in existing_dates:
            url = f"{BASE_URL}/{fname}"
            new_rows.append({
                "date": date, "res":RES, 
                "country":COUNTRY, "url": url,
                "carbon_mean": CARBON_MEAN, "land_area": LAND_AREA,
                "ol": OL, "s": S, "yrf": YRF, "ldf": LDF, "mdf": MDF,
                "hdf": HDF, "total": TOTAL
            })

if new_rows:
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.sort_values("date").reset_index(drop=True)

    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, float_format='%.2f', index=False)
    blob_csv.upload_from_string(csv_buffer.getvalue(), content_type="text/csv")

    print(f"{len(new_rows)} new COG added to {CSV_BLOB_NAME}")
else:
    print("No new COG detected in the bucket.")


## Geojsons creation and upload

In [ ]:
def reproject_if_needed(gdf_arg, target_crs):
    """Reprojects a GeoDataFrame to the target CRS if necessary."""
    if gdf_arg.crs is None:
        print("[WARNING] GeoJSON has no CRS defined.")
        gdf_arg = gdf_arg.set_crs(CRSG)

    if gdf_arg.crs != target_crs:
        print(f"Reprojecting: From {gdf_arg.crs} to {target_crs}")
        gdf_arg = gdf_arg.to_crs(target_crs)
    else:
        print(f"CRS matches ({target_crs}), no reprojection needed.")
    return gdf_arg

def compute_zonal_stats(gdf_arg, raster_path, band=1, nodata=NODATA_VAL):
    """Computes zonal statistics for each polygon in the GeoDataFrame."""

    stats_needed = ["mean", "min", "max", "median", "std", "count"]
    print(f"\nComputing zonal statistics (band {band})...")
    results = zonal_stats(
        gdf_arg,
        raster_path,
        band=band,
        stats=stats_needed,
        nodata=nodata,
        all_touched=False,   # set to True to include partially covered pixels
        geojson_out=False,
        masked=True
    )
    return results

def update_geodataframe(gdf_arg, stats_results):
    """ Adds zonal statistics columns to the GeoDataFrame. """
    column_mapping = {
        "mean":   "carbon_mean",
        "min":    "carbon_min",
        "max":    "carbon_max",
        "median": "carbon_median",
        "std":    "carbon_std",
        "count":  "carbon_pixel_count",
    }

    for stat_key, col_name in column_mapping.items():
        gdf_arg[col_name] = [
            round(r[stat_key], 2) if r[stat_key] is not None else None
            for r in stats_results
        ]
    return gdf_arg


In [ ]:
def run_zonal_stats(raster_path, geojson_path, output_path, col, band=1, nodata=NODATA_VAL):
    """
    Performs zonal statistics on a raster using a GeoJSON of polygons,
    adds the statistics as new columns, and saves the updated GeoJSON.
    """
    # Read raster metadata
    print(f"\nReading raster: {raster_path}")
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        raster_nodata = src.nodata if nodata is None else nodata
        print(f"Raster CRS   : {raster_crs}")
        print(f"Dimensions   : {src.width} x {src.height} pixels")
        print(f"Bands        : {src.count}")
        print(f"NoData value : {raster_nodata}")

    # Read GeoJSON
    print(f"\nReading GeoJSON: {geojson_path}")
    gdf_in = gpd.read_file(geojson_path)
    print(f"Loaded {len(gdf_in)} polygons")
    print(f"Existing columns: {list(gdf_in.columns)}")

    # Reproject if necessary
    print("\nChecking projection...")
    gdf_in = reproject_if_needed(gdf_in, raster_crs)

    # Compute zonal statistics
    print("\nExtracting zonal statistics...")
    stats_results = compute_zonal_stats(gdf_in, raster_path, band=band, nodata=raster_nodata)
    gdf_in = update_geodataframe(gdf_in, stats_results)

    # Preview and save
    print("\nPreview of results:")
    preview_cols = [col, "carbon_mean", "carbon_min", "carbon_max",
                    "carbon_median", "carbon_std", "carbon_pixel_count"]
    # Keep only columns that actually exist
    preview_cols = [c for c in preview_cols if c in gdf_in.columns]
    print(gdf_in[preview_cols].to_string(index=False))

    gdf_in.to_file(output_path, driver="GeoJSON")
    print(f"\nUpdated GeoJSON saved to: {output_path}")

    return gdf_in


### Regions

In [ ]:
regions_gdf = run_zonal_stats(
    raster_path=COG_OUTPUT_PATH,
    geojson_path=BASE_REGIONS_PATH,
    output_path=OUT_REGIONS_PATH,
    col="NAME_1",
    band=1
)


### Departments

In [ ]:
departments_gdf = run_zonal_stats(
    raster_path=COG_OUTPUT_PATH,
    geojson_path=BASE_DEPTS_PATH,
    output_path=OUT_DEPTS_PATH,
    col="NAME_2",
    band=1
)


### Communes

In [ ]:
communes_gdf = run_zonal_stats(
    raster_path=COG_OUTPUT_PATH,
    geojson_path=BASE_COMS_PATH,
    output_path=OUT_COMS_PATH,
    col="NAME_4",
    band=1
)


### Protected Areas

In [ ]:
protected_areas_gdf = run_zonal_stats(
    raster_path=COG_OUTPUT_PATH,
    geojson_path=BASE_PA_PATH,
    output_path=OUT_PA_PATH,
    col="NAME",
    band=1
)


### Upload

In [ ]:
for filepath in glob.glob(f"{GEOJSON_DIR}/*.geojson"):
    filename = os.path.basename(filepath)
    if "regions" in filename:
        destination_blob = f"{GCS_REGIONS_PREFIX}/{filename}"
    elif "departments" in filename:
        destination_blob = f"{GCS_DEPTS_PREFIX}/{filename}"
    elif "communes" in filename:
        destination_blob = f"{GCS_COMS_PREFIX}/{filename}"
    elif "protected_areas" in filename:
        destination_blob = f"{GCS_PA_PREFIX}/{filename}"
    else:
        continue
    blob_dst = bucket.blob(destination_blob)
    try:
        blob_dst.upload_from_filename(filepath, content_type='application/geo+json')
    except Exception as e:
        print(f"Error uploading {filepath} to {destination_blob}: {e}")


## CSVs Creation

### Regions

In [ ]:
regions_df = pd.DataFrame(gpd.read_file(OUT_REGIONS_PATH))
regions_df.insert(REG_POS, "date_year_month", PERIOD)
regions_df.to_csv(f"{CSV_DIR}/senegal_regions_preds_{PERIOD}.csv", index=False)



### Departments

In [ ]:
departments_df = pd.DataFrame(gpd.read_file(OUT_DEPTS_PATH))
departments_df.insert(DEPT_POS, "date_year_month", PERIOD)
departments_df.to_csv(f"{CSV_DIR}/senegal_departments_preds_{PERIOD}.csv", index=False)


### Communes

In [ ]:
communes_df = pd.DataFrame(gpd.read_file(OUT_COMS_PATH))
communes_df.insert(COM_POS, "date_year_month", PERIOD)
communes_df.to_csv(f"{CSV_DIR}/senegal_communes_preds_{PERIOD}.csv", index=False)


### Protected Areas

In [ ]:
protected_areas_df = pd.DataFrame(gpd.read_file(OUT_PA_PATH))
protected_areas_df.insert(PA_POS, "date_year_month", PERIOD)
protected_areas_df.to_csv(f"{CSV_DIR}/senegal_protected_areas_preds_{PERIOD}.csv", index=False)


### Upload

In [ ]:
for filepath in glob.glob(f"{CSV_DIR}/*.csv"):
    filename = os.path.basename(filepath)
    if "regions" in filename:
        destination_blob = f"{GCS_REGIONS_PREFIX}/{filename}"
    elif "departments" in filename:
        destination_blob = f"{GCS_DEPTS_PREFIX}/{filename}"
    elif "communes" in filename:
        destination_blob = f"{GCS_COMS_PREFIX}/{filename}"
    elif "protected_areas" in filename:
        destination_blob = f"{GCS_PA_PREFIX}/{filename}"
    else:
        continue
    blob_dst = bucket.blob(destination_blob)
    try:
        blob_dst.upload_from_filename(filepath, content_type='text/csv')
    except Exception as e:
        print(f"Error uploading {filepath} to {destination_blob}: {e}")


## Raw Tiles Cleaning

In [ ]:
assert blob_cog.exists(), "COG does not exist yet !"
print(f"COG in GCS : gs://{BUCKET_NAME}/{GCS_COG_PATH}")

raw_blobs = list(bucket.list_blobs(prefix=GCS_TILES_PREFIX + "/"))
print(f"\nCleaning of {len(raw_blobs)} raw tiles in GCS ...")
print(f"Prefix : gs://{BUCKET_NAME}/{GCS_TILES_PREFIX}/")

deleted = 0
errors = []

for blob in tqdm(raw_blobs, desc="GCS Cleaning", unit="file"):
    try:
        blob.delete()
        deleted += 1
    except Exception as e:
        errors.append((blob.name, str(e)))

print(f"\n{'─'*50}")
print(f"{deleted} raw tiles removed from GCS.")
if errors:
    print(f"{len(errors)} errors :")
    for name, err in errors:
        print(f"{name} : {err}")
print(f"{'─'*50}")

local_pred_files = list(Path(LOCAL_PRED_DIR).glob("*.tif"))
for f in local_pred_files:
    f.unlink()
print(f"\n{len(local_pred_files)} local prediction tiles removed.")


## Geojsons and CSVs Cleaning

In [ ]:
for filepath in glob.glob(f"{GEOJSON_DIR}/*.geojson"):
    os.remove(filepath)
print(f"\n{len(glob.glob(f'{GEOJSON_DIR}/*.geojson'))} local GeoJSON files removed.")

for filepath in glob.glob(f"{CSV_DIR}/*.csv"):
    os.remove(filepath)
print(f"\n{len(glob.glob(f'{CSV_DIR}/*.csv'))} local CSV files removed.")
